### Created on 10/20/2025 by MTH 
# Notebook to use to generate csv input files for the MCMC model 
The idea with this notebook is that is should be a clear way to generate and save new CSV files to read into the MCMC model without just copying and pasting some messy CSV files and notes. 


In [2]:
# Import pytorch libraries 
import numpy as np 
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

import sys
import os
import glob
from tqdm import tqdm 

from scipy.optimize import curve_fit


# Add the directory containing the module to sys.path
module_path = os.path.abspath("/crucial/modified_MCMC/dATP_multiscale_modeling/resultsOrg")
if module_path not in sys.path:
    sys.path.append(module_path)

# Import the module
import helper_functions as hf

In [3]:
parameter_set_ABC = pd.read_csv('/crucial/modified_MCMC/dATP_multiscale_modeling/MCMC_simulation_results/2025-09-25_1038/baseline_parameter_set_10132025.csv', comment = '#', 
    nrows = 3, 
    skipinitialspace = True)

parameter_set_ABC

parameter_set_ABC.rename(columns= {"K_SS":"k_plus_SS"}, inplace = True)

parameter_set_ABC['k_plus_SS'] = 0.02
parameter_set_ABC['k_minus_SS'] = parameter_set_ABC['k_plus_SS'] / 3.59 

parameter_set_ABC['K_D'] = 0.5934
parameter_set_ABC['coop_N'] = 1.0


previous_csv = parameter_set_ABC.iloc[2, :].to_frame().T

previous_csv

,protocol,k_force_baseline,k_force_drug,k_plus_SR_baseline,k_plus_SR_drug,k_minus_SR,k_xb,k1_plus_ref_baseline,k1_plus_ref_drug,k2_plus_baseline,...,delta_G_ATP,alpha,beta,eta,g_Cb,g_Ca,k_plus_SS,k_minus_SS,K_D,coop_N
2,1.0,0.2,779.0,16.0,16.0,17.580187,5.0,0.005017,0.00478,0.024486,...,-13.0,0.28,0.35,0.68,0.0,1.0,0.02,0.005571,0.5934,1.0


In [4]:

# Make sure that all the druc rates match between baseline and drug 
for i in range(1,len(previous_csv.columns)):
    if 'drug' in previous_csv.columns[i]:
        if 'baseline' in previous_csv.columns[i -1]:
            print("Checkout out for column ", previous_csv.columns[i], " and ", previous_csv.columns[i-1])
            if previous_csv.iloc[0,i] != previous_csv.iloc[0,i-1]:
                print("Values don't match! ", previous_csv.iloc[0,i], " and ", previous_csv.iloc[0,i-1])
                print("Setting values to match now: ")
                previous_csv.iloc[0,i] = previous_csv.iloc[0,i-1]
                print("New values: ", previous_csv.iloc[0,i], " and ", previous_csv.iloc[0,i-1])
        else: 
            print("Didn't checkout for column ", previous_csv.columns[i], " and ", previous_csv.columns[i-1])

Checkout out for column  k_force_drug  and  k_force_baseline
Values don't match!  779.0  and  0.2
Setting values to match now: 
New values:  0.2  and  0.2
Checkout out for column  k_plus_SR_drug  and  k_plus_SR_baseline
Checkout out for column  k1_plus_ref_drug  and  k1_plus_ref_baseline
Values don't match!  0.00478  and  0.0050173923373222
Setting values to match now: 
New values:  0.0050173923373222  and  0.0050173923373222
Checkout out for column  k2_plus_drug  and  k2_plus_baseline
Values don't match!  0.0015  and  0.0244862269610166
Setting values to match now: 
New values:  0.0244862269610166  and  0.0244862269610166
Checkout out for column  k3_plus_drug  and  k3_plus_baseline
Values don't match!  0.08  and  0.0141926184296607
Setting values to match now: 
New values:  0.0141926184296607  and  0.0141926184296607
Checkout out for column  k4_plus_ref_drug  and  k4_plus_ref_baseline
Values don't match!  0.23  and  0.0356220044195652
Setting values to match now: 
New values:  0.03562

In [5]:
# Argument dictionary witll be set as some 


def add_parameter_experiments(df, experiments):
    """
    Add new parameter experiments to the dataframe.
    
    Parameters:
    -----------
    df : pd.DataFrame
        The existing dataframe with MCMC parameters
    experiments : dict or list of dicts
        Single dict or list of dicts containing parameter values to modify.
        Only include parameters you want to change from the baseline (row 0).
        
    Returns:
    --------
    pd.DataFrame
        Updated dataframe with new experiments appended
        
    Examples:
    ---------
    # Single experiment, changing one parameter
    df = add_parameter_experiments(df, {'k_force_baseline': 0.3})
    
    # Single experiment, changing multiple parameters
    df = add_parameter_experiments(df, {
        'k_force_baseline': 0.3,
        'k_force_drug': 0.25,
        'percent_drug': 0.5
    })
    
    # Multiple experiments at once
    df = add_parameter_experiments(df, [
        {'k_force_baseline': 0.3},
        {'k_force_baseline': 0.4},
        {'k_force_baseline': 0.5, 'percent_drug': 0.3}
    ])
    """
    # Convert single dict to list for uniform processing
    if isinstance(experiments, dict):
        experiments = [experiments]
    
    # Get baseline parameters (assumes row 0 is baseline)
    baseline = df.iloc[0].to_dict()
    
    # Create new rows
    new_rows = []
    for exp in experiments:
        # Start with baseline values
        new_row = baseline.copy()
        # Update with new parameter values
        new_row.update(exp)
        new_rows.append(new_row)
    
    # Create new dataframe with new rows
    new_df = pd.DataFrame(new_rows)
    
    # Append to original dataframe and reset index
    result_df = pd.concat([df, new_df], ignore_index=True)
    
    return result_df


# Alternative: Generate grid of parameter combinations
def add_parameter_grid(df, param_grid):
    """
    Add experiments for all combinations of parameter values (grid search).
    
    Parameters:
    -----------
    df : pd.DataFrame
        The existing dataframe with MCMC parameters
    param_grid : dict
        Dictionary mapping parameter names to lists of values to try
        
    Returns:
    --------
    pd.DataFrame
        Updated dataframe with all parameter combinations appended
        
    Example:
    --------
    df = add_parameter_grid(df, {
        'k_force_baseline': [0.2, 0.3, 0.4],
        'percent_drug': [0.0, 0.5, 1.0]
    })
    # This creates 3 x 3 = 9 new experiments
    """
    import itertools
    
    # Get all combinations
    param_names = list(param_grid.keys())
    param_values = list(param_grid.values())
    combinations = list(itertools.product(*param_values))
    
    # Create experiment dicts
    experiments = [dict(zip(param_names, combo)) for combo in combinations]
    
    return add_parameter_experiments(df, experiments)



In [11]:
previous_csv['k2_plus_drug'].values
reduction_rate = [0.7,0.65,0.6]
k2_drug_rates = previous_csv['k2_plus_baseline'].values * reduction_rate
k2_drug_rates

array([0.01714036, 0.01591605, 0.01469174])

In [13]:
previous_csv['k_D'] = 6.817

drug_conc = add_parameter_grid(previous_csv, {
    'K_D': [6.817],
    'k_plus_SS': [0.02],
    'k2_plus_drug': k2_drug_rates,
    'coop_N': [1],
    'percent_drug': [0.3, 1, 3, 30],
})

drug_conc# .to_csv('temp_output/pset_C_K_D_sweep_fit_coop_exploration.csv', index=False)


,protocol,k_force_baseline,k_force_drug,k_plus_SR_baseline,k_plus_SR_drug,k_minus_SR,k_xb,k1_plus_ref_baseline,k1_plus_ref_drug,k2_plus_baseline,...,alpha,beta,eta,g_Cb,g_Ca,k_plus_SS,k_minus_SS,K_D,coop_N,k_D
0,1.0,0.2,0.2,16.0,16.0,17.580187,5.0,0.005017,0.005017,0.024486,...,0.28,0.35,0.68,0.0,1.0,0.02,0.005571,0.5934,1.0,6.817
1,1.0,0.2,0.2,16.0,16.0,17.580187,5.0,0.005017,0.005017,0.024486,...,0.28,0.35,0.68,0.0,1.0,0.02,0.005571,6.8170,1.0,6.817
2,1.0,0.2,0.2,16.0,16.0,17.580187,5.0,0.005017,0.005017,0.024486,...,0.28,0.35,0.68,0.0,1.0,0.02,0.005571,6.8170,1.0,6.817
3,1.0,0.2,0.2,16.0,16.0,17.580187,5.0,0.005017,0.005017,0.024486,...,0.28,0.35,0.68,0.0,1.0,0.02,0.005571,6.8170,1.0,6.817
4,1.0,0.2,0.2,16.0,16.0,17.580187,5.0,0.005017,0.005017,0.024486,...,0.28,0.35,0.68,0.0,1.0,0.02,0.005571,6.8170,1.0,6.817
5,1.0,0.2,0.2,16.0,16.0,17.580187,5.0,0.005017,0.005017,0.024486,...,0.28,0.35,0.68,0.0,1.0,0.02,0.005571,6.8170,1.0,6.817
6,1.0,0.2,0.2,16.0,16.0,17.580187,5.0,0.005017,0.005017,0.024486,...,0.28,0.35,0.68,0.0,1.0,0.02,0.005571,6.8170,1.0,6.817
7,1.0,0.2,0.2,16.0,16.0,17.580187,5.0,0.005017,0.005017,0.024486,...,0.28,0.35,0.68,0.0,1.0,0.02,0.005571,6.8170,1.0,6.817
8,1.0,0.2,0.2,16.0,16.0,17.580187,5.0,0.005017,0.005017,0.024486,...,0.28,0.35,0.68,0.0,1.0,0.02,0.005571,6.8170,1.0,6.817
9,1.0,0.2,0.2,16.0,16.0,17.580187,5.0,0.005017,0.005017,0.024486,...,0.28,0.35,0.68,0.0,1.0,0.02,0.005571,6.8170,1.0,6.817


In [14]:
if input("Want to save this? (y/n) ") == 'y':
    fname = input("Write the filename you want to save as (include the .csv extension): ")
    drug_conc.to_csv(f'temp_output/{fname}', index=False)